In [1]:
import pandas as pd 
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
53,83,Male,98.0,Mild,Delhi,Yes
39,50,Female,103.0,Mild,Kolkata,No
66,51,Male,104.0,Mild,Kolkata,No
70,68,Female,101.0,Strong,Delhi,No
23,80,Female,98.0,Mild,Delhi,Yes


In [5]:
from sklearn.model_selection import train_test_split
x_train , x_test , y_train ,y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=0)

In [6]:
#adding simple imputer to fever column
si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])

x_test_fever = si.fit_transform(x_test[['fever']])

In [7]:
# ordinal encoind for cough column
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])
x_test_cough = oe.fit_transform(x_test[['cough']])


In [8]:
#one hot encoding omn cities and gender 

ohe = OneHotEncoder(drop='first')
x_train_city_gender = ohe.fit_transform(x_train[['city', 'gender']])

x_test_city_gender = ohe.fit_transform(x_test[['city', 'gender']])

In [9]:
#Extracting age 

x_train_age = x_train.drop(columns=['gender'	,'fever',	'cough',	'city'	])

x_test_age = x_test.drop(columns=['gender'	,'fever',	'cough',	'city'	])

In [10]:
print(x_train_age.shape)
print(x_train_fever.shape)
print(x_train_city_gender.shape)
print(x_train_cough.shape)

print(x_test_age.shape)
print(x_test_fever.shape)
print(x_test_city_gender.shape)
print(x_test_cough.shape)


(80, 1)
(80, 1)
(80, 4)
(80, 1)
(20, 1)
(20, 1)
(20, 4)
(20, 1)


In [11]:
# we used to array bcz by default its <class 'scipy.sparse._csr.csr_matrix'> which is sparse matrix and the contact doesnt work on the matrix soo we convert it to array and then we can concatenate it with other features

x_train_city_gender = x_train_city_gender.toarray()
x_test_city_gender = x_test_city_gender.toarray()

In [15]:
x_train_transformed = np.concatenate((x_train_age, x_train_fever, x_train_city_gender, x_train_cough), axis=1)
x_test_transformed = np.concatenate((x_test_age, x_test_fever, x_test_city_gender, x_test_cough), axis=1)  

x_train_transformed.shape


(80, 7)

### using  column transformer 

In [16]:
from sklearn.compose import ColumnTransformer

In [19]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']), 
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(drop='first'), ['gender', 'city'])
    ],
    remainder='passthrough'
)

In [21]:
transformer.fit_transform(x_train).shape

(80, 7)

In [23]:
transformer.transform(x_test).shape

(20, 7)